# Colab: original và MoE classification head

Chạy từng bước. Notebook chỉ gọi pipeline chung; không tự đánh giá test.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Dùng repo đã clone/upload; chọn đường dẫn phù hợp phiên làm việc.
REPO_DIR = Path("/content/code") if "COLAB_RELEASE_TAG" in os.environ else Path("/kaggle/working/code")
if Path("train.py").exists():
    REPO_DIR = Path.cwd()
REPOSITORY_URL = "https://github.com/lngphgthao/moe-vihsd.git"
BRANCH = "classifier-main"  # Cần push nhánh đã refactor trước khi clone từ remote.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(REPO_DIR)], check=True)
assert (REPO_DIR / "train.py").is_file(), "Đặt REPO_DIR đến checkout/upload chứa code mới"
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"Chưa lấy được secret {key}; thêm vào Colab Secrets nếu cần")
os.environ["RUNS_DIR"] = "/content/drive/MyDrive/thesis/runs"


In [ ]:
# CI chạy unittest offline trong GitHub Actions.
# Notebook chỉ chạy experiment; không lặp toàn bộ test suite.


## Chọn experiment

exp-003/004: ViANLI original/MoE; exp-005/006: ViHSD original/MoE.
PhoBERT cần khôi phục hoặc tạo segmented dataset theo data/manifests trước khi train.


In [ ]:
CONFIG = "configs/experiments/vianli/exp-003.yaml"
SMOKE_TEST = True
# Chọn ID chưa dùng; tracker CSV và W&B liên kết cùng run ID.
RUN_ID = "run-010"  # Ví dụ smoke exp-003; đổi ID nếu đã dùng.
command = [sys.executable, "train.py", "--config", CONFIG, "--run-id", RUN_ID,
           "--smoke-test" if SMOKE_TEST else "--no-smoke-test"]
subprocess.run(command, check=True)


In [ ]:
subprocess.run([sys.executable, "scripts/aggregate_results.py", "--runs-dir", os.environ["RUNS_DIR"],
                "--output-dir", str(Path(os.environ["RUNS_DIR"]) / "analysis")], check=True)


## Test sau khi khóa finalist

Chỉ chạy evaluate khi config và seed đã khóa; lệnh mẫu bên dưới để chủ động sửa/chạy.
Giữ toàn bộ run directory, tracker CSV và W&B URL trước khi kết thúc phiên GPU.


In [ ]:
# subprocess.run([sys.executable, "evaluate.py", "--run-dir", "<run directory>",
#                 "--finalist-config", "<locked finalist YAML>", "--final-evaluation"], check=True)
